# lineare model

In [8]:
library(ggplot2)
library(lattice)
library(caret)

In [9]:
train_values <- read.csv("train_values.csv",stringsAsFactors = T)
test_values <- read.csv("test_values.csv",stringsAsFactors = T)
train_labels <- read.csv("train_labels.csv",stringsAsFactors = T)
submission_format <- read.csv("submission_format.csv",stringsAsFactors = T)

In [91]:
data <- merge(train_values,train_labels,by=c('building_id','building_id'),all.x=T)
data <- data[,-1]
data[39] <- lapply(data[39], function(x) as.character(x))
data[,39]<-factor(data[,39])
dummy_data <- dummyVars(" ~ .", data=data)  
data <- data.frame(predict(dummy_data, newdata = data)) 
data <- data[,-c(8,11,17,20,24,28,32,53)]

In [92]:
dim(data)

[1] 260601     63

### Zero- and Near Zero-Variance Predictors

In [96]:
nzv <- nearZeroVar(data)
data <- data[, -nzv]

### Identifying Correlated Predictors

In [99]:
dataCor <- cor(data[,1:29])
highlyCordata <- findCorrelation(dataCor, cutoff = .75)
data <- data[,-highlyCordata]

### Linear Regression of an Indicator Matrix

In [171]:
k = 5
target_variable <- 28:30
F_micro <- array(0,k)
threshold <- 0.5

Shuffle_idx <- sample(1:260601)
max <- ceiling(260601/k)
splits <- split(Shuffle_idx, ceiling(seq_along(Shuffle_idx)/max))

for (i in 1:k){
    test_data <- data[splits[[i]],]
    train_data <- data[-splits[[i]],]
    
    X <- as.matrix(test_data[,setdiff(colnames(test_data),c("damage_grade.1","damage_grade.2","damage_grade.3"))])
    Y <- as.matrix(test_data[,c("damage_grade.1","damage_grade.2","damage_grade.3")])
    nr_X = nrow(X)
    X <- cbind(matrix(1, nr = nr_X, nc = 1),X)
    beta_hat <- solve(t(X)%*%X)%*%t(X)%*%Y
    
    A <- as.matrix(train_data[,setdiff(colnames(train_data),c("damage_grade.1","damage_grade.2","damage_grade.3"))])
    B <- as.matrix(train_data[,c("damage_grade.1","damage_grade.2","damage_grade.3")])
    
    nr = nrow(A)
    sol <- cbind(matrix(1, nr = nr, nc = 1),A) %*% beta_hat
    
    cfm <- matrix(0, nrow = 3, ncol = 3)
    for (j in 1:nr){
        cfm[which.max(sol[j,]),which.max(B[j,])] <- cfm[which.max(sol[j,]),which.max(B[j,])]+1
    }
    TP <- c(cfm[1,1],cfm[2,2],cfm[3,3])
    FP <- c(cfm[2,1]+cfm[3,1],cfm[1,2]+cfm[3,2],cfm[1,3]+cfm[2,3])
    FN <- c(cfm[1,2]+cfm[1,3],cfm[2,1]+cfm[2,3],cfm[3,1]+cfm[3,2])
    P_micro <- sum(TP)/sum(TP+FP)
    R_micro <- sum(TP)/sum(TP+FN)
    F_micro[i] = (2*P_micro*R_micro)/(P_micro+R_micro)
}
print(mean(F_micro))

[1] 0.5798443
